In [221]:
import pandas as pd
file_path ='MRNGO_features.xlsx'

df1 = pd.read_excel(file_path, sheet_name='features')
df2 = pd.read_excel(file_path, sheet_name='concept_features')

merged_df = pd.merge(df1, df2, on='Feature', how='left')

file_path = 'MRNGO_features_clustered_stimulimaking_preprocessed.xlsx'
df3 = pd.read_excel(file_path)

df_final = pd.merge(merged_df, df3, left_on='Feature', right_on='vector_lemma_C', how='inner')

cols = list(df_final.columns)
cols.insert(cols.index('Feature') + 1, cols.pop(cols.index('vector_lemma_C')))
df_final = df_final[cols]

cols_to_drop = set()
for i in range(len(df_final.columns)):
    col_name_1 = df_final.columns[i]
    if col_name_1 in cols_to_drop:
        continue
    for j in range(i + 1, len (df_final.columns)):
        col_name_2 = df_final.columns[j]
        if df_final.iloc[:, i].equals(df_final.iloc[:, j]):
            cols_to_drop.add(col_name_2)
df_final = df_final.drop(columns=cols_to_drop)

df_final.to_excel('pilot_features.xlsx', sheet_name='sorted_by_features', index=False)


In [192]:
import pandas as pd
import numpy as np
import os 

file_path = 'pilot_features.xlsx'
sheet_name = 'sorted_by_features'
column_to_check = 'itemno'
new_column_name = 'filter_status'

# features that we did not visualized at the end 
items_to_exclude = [ 
    'f00014',
    'f00213',
    'f00217',
    'f00220',
    'f00486',
    'f00497',
    'f00501',
    'f00649',
    'f00991',
    'f01105',
    'f01406',
    'f01528',
    'f01932',
    'f02259'
]

if not os.path.exists(file_path):
    print(f"file was not find")
else:
    try:
        df = pd.read_excel(file_path, sheet_name=sheet_name)
        df[new_column_name] = np.where(
            df[column_to_check].isin(items_to_exclude),
            'excluded',
            'included'
        )

        with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
            df.to_excel(writer,sheet_name=sheet_name, index=False)
        print(f"success, sheet has been updated")

    except PermissionError:
        print(f"permission error")

success, sheet has been updated


In [193]:
import pandas as pd
file_path = 'pilot_features.xlsx'
sheet_name = 'sorted_by_features'
df = df[(df['filter_status'] == 'included') & (df['Feature'] != 'HOL_BOLT')]

# filtering out features that were excluded from the set and also the feature 'HOL_BOLT'


In [194]:
df_freq_x_top10 = df.groupby('Concept').apply(
    lambda x: x.nlargest(10, 'frequency_x')
).reset_index(drop=True)
df_sequences1 = df_freq_x_top10.groupby('Conceptno')['Feature'].apply(list).reset_index()

In [195]:
df_freq_x_low10 = df.groupby('Concept').apply(
    lambda x: x.nsmallest(10, 'frequency_x')
).reset_index(drop=True)
df_sequences2 = df_freq_x_low10.groupby('Conceptno')['Feature'].apply(list).reset_index()

In [196]:
df_freq_y_top10 = df.groupby('Concept').apply(
    lambda x: x.nlargest(10, 'frequency_y')
).reset_index(drop=True)
df_sequences3 = df_freq_y_top10.groupby('Conceptno')['Feature'].apply(list).reset_index()

In [197]:
df_freq_y_low10 = df.groupby('Concept').apply(
    lambda x: x.nsmallest(10, 'frequency_y')
).reset_index(drop=True)
df_sequences4 = df_freq_y_low10.groupby('Conceptno')['Feature'].apply(list).reset_index()

In [198]:
df_dist_top10 = df.groupby('Concept').apply(
    lambda x: x.nlargest(10, 'Distinct')
).reset_index(drop=True)
df_sequences5 = df_dist_top10.groupby('Conceptno')['Feature'].apply(list).reset_index()

In [199]:
df_dist_low10 = df.groupby('Concept').apply(
    lambda x: x.nsmallest(10, 'Distinct')
).reset_index(drop=True)
df_sequences6 = df_dist_low10.groupby('Conceptno')['Feature'].apply(list).reset_index()

In [200]:
df_random = df.groupby('Concept').apply(
    lambda x: x.sample(n=min(len(x), 10))
).reset_index(drop=True)
df_sequences7 = df_random.groupby('Conceptno')['Feature'].apply(list).reset_index()

In [201]:
data_frames = [df_sequences1, df_sequences2, df_sequences3, df_sequences4, df_sequences5, df_sequences6, df_sequences7]
for i, df in enumerate(data_frames, 1):
    other_col = [c for c in df.columns if c != 'Conceptno'][0]
    df.rename(columns={other_col: f'sequence_{i}'}, inplace=True)

from functools import reduce
df_merged = reduce(lambda left, right: pd.merge(left, right, on='Conceptno', how='outer'), data_frames)

In [202]:
data_frames = [df_sequences1, df_sequences2, df_sequences3, df_sequences4, df_sequences5, df_sequences6, df_sequences7]
new_names = ['freq_x_top10', 'freq_x_low10', 'freq_y_top10', 'freq_y_low10', 'dist_top10', 'dist_low10', 'random10']

for df, name in zip(data_frames, new_names):
    col_to_change = [c for c in df.columns if c != 'Conceptno'][0]
    df.rename(columns={col_to_change: name}, inplace=True)

In [203]:
from functools import reduce
df_merged = reduce(lambda left, right: pd.merge(left, right, on='Conceptno', how='outer'), data_frames)
df_merged.head()


,Conceptno,freq_x_top10,freq_x_low10,freq_y_top10,freq_y_low10,dist_top10,dist_low10,random10
0,a05,"[RÉSZE_LÁB, ILYEN_NAGY, ILYEN_BARNA, FAJTA_ÁLL...","[HOL_ÁLLATKERT, ILYEN_ALACSONY, HOL_MESE, ILYE...","[RÉSZE_LÁB, FAJTA_ÁLLAT, ILYEN_BARNA, ILYEN_SÁ...","[RÉSZE_UJJ, RÉSZE_FAROK, ALFAJTA_ZÖLD, ILYEN_K...","[RÉSZE_FAROK, ILYEN_CUKI, HOL_ÁLLATKERT, ILYEN...","[ILYEN_NAGY, ILYEN_BARNA, ALFAJTA_KICSI, HOL_O...","[ILYEN_SÁRGA, TUD_MEGY, HOL_MESE, ALFAJTA_KICS..."
1,a16,"[RÉSZE_LÁB, ILYEN_NAGY, ILYEN_BARNA, HOL_HÁZ, ...","[ILYEN_BOLDOG, ILYEN_HANGOS, RÉSZE_HAS, TUD_SÉ...","[RÉSZE_FAROK, RÉSZE_LÁB, FAJTA_ÁLLAT, NÉGY_LÁB...","[ILYEN_FINOM, ILYEN_HOSSZÚ, ILYEN_KICSI, LEHET...","[RÉSZE_FAROK, NÉGY_LÁB, RÉSZE_FÜL, RÉSZE_HAJ, ...","[ILYEN_NAGY, HOL_HÁZ, ILYEN_BARNA, ILYEN_HOSSZ...","[TUD_FUT, RÉSZE_FAROK, ILYEN_HANGOS, RÉSZE_FEJ..."
2,a19,"[RÉSZE_LÁB, ILYEN_NAGY, ILYEN_BARNA, FAJTA_ÁLL...","[HOL_ÁLLATKERT, ILYEN_BOLDOG, RÉSZE_HAS, TUD_S...","[FAJTA_ÁLLAT, RÉSZE_FAROK, LEHET_LOVAGOL, RÉSZ...","[ILYEN_KICSI, HOL_PIAC, ILYEN_PUHA, LEHET_JÁR,...","[RÉSZE_FAROK, NÉGY_LÁB, LEHET_LOVAGOL, RÉSZE_F...","[ILYEN_NAGY, ILYEN_BARNA, HOL_PIAC, ILYEN_KICS...","[RÉSZE_NYAK, RÉSZE_HAS, RÉSZE_TEST, ILYEN_BARN..."
3,b02,"[ILYEN_BARNA, ILYEN_PIROS, LEHET_OLVAS, ALFAJT...","[LEHET_NÉZ, RÉSZE_KÖR, TUD_SEGÍT, HOL_MESE, IL...","[ILYEN_BARNA, ILYEN_KÉK, ALFAJTA_ZÖLD, EMBER_R...","[RÉSZE_BELSŐ, LEHET_JÁTSZIK, ILYEN_FEHÉR, BENN...","[EMBER_RÉSZE, TUD_SEGÍT, LEHET_NÉZ, RÉSZE_KÖR,...","[ILYEN_BARNA, RÉSZE_BELSŐ, LEHET_JÁTSZIK, KI_E...","[ILYEN_PIROS, BENNE_EMBER, KI_EMBER, LEHET_JÁT..."
4,b10,"[LEHET_ESZIK, ILYEN_NAGY, RÉSZE_UJJ, ILYEN_HOS...","[LEHET_MOZOG, RÉSZE_HÚS, RÉSZE_BŐR, RÉSZE_KÖR,...","[RÉSZE_UJJ, RÉSZE_KÖRÖM, EMBER_RÉSZE, LEHET_FO...","[ILYEN_NAGY, ILYEN_HOSSZÚ, LEHET_ÍR, LEHET_FEL...","[EMBER_RÉSZE, LEHET_FOG, ÖT_UJJ, TUD_RAJZOL, R...","[ILYEN_NAGY, ILYEN_HOSSZÚ, LEHET_JÁTSZIK, KI_E...","[KI_EMBER, TUD_RAJZOL, HOL_TEST, LEHET_TESZ, I..."


In [204]:
import pandas as pd

for col in new_names:
    df_merged[col] = df_merged[col].apply(lambda x: ', '.join(map(str, x)) if isinstance(x, list) else x)
with pd.ExcelWriter('pilot_features.xlsx', engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df_merged.to_excel(writer, sheet_name='sequences_feature', index=False)
print

<function print(*args, sep=' ', end='\n', file=None, flush=False)>

In [205]:
import pandas as pd
import numpy as np
file_path = 'pilot_features.xlsx'
sheet_name = 'sorted_by_features'
column_to_check = 'itemno'
new_column_name = 'filter_status'

# features that we did not visualized at the end 
items_to_exclude = [ 
    'f00014',
    'f00213',
    'f00217',
    'f00220',
    'f00486',
    'f00497',
    'f00501',
    'f00649',
    'f00991',
    'f01105',
    'f01406',
    'f01528',
    'f01932',
    'f02259'
]

df = pd.read_excel(file_path, sheet_name=sheet_name)
df[new_column_name] = np.where(
    df[column_to_check].isin(items_to_exclude),
    'excluded',
    'included'
)

try:
    with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        df.to_excel(writer,sheet_name=sheet_name, index=False)
    print(f"success, sheet has been updated")

except PermissionError:
    print(f"error")

success, sheet has been updated


In [206]:
import pandas as pd
file_path = 'pilot_features.xlsx'
sheet_name = 'sorted_by_features'
df = df[(df['filter_status'] == 'included') & (df['Feature'] != 'HOL_BOLT')]

# filtering out features that were excluded from the set and also the feature 'HOL_BOLT'


In [207]:
df_freq_x_top10_2 = df.groupby('Concept').apply(
    lambda x: x.nlargest(10, 'frequency_x')
).reset_index(drop=True)
df_sequences8 = df_freq_x_top10_2.groupby('Conceptno')['itemno'].apply(list).reset_index()

df_freq_x_low10_2 = df.groupby('Concept').apply(
    lambda x: x.nsmallest(10, 'frequency_x')
).reset_index(drop=True)
df_sequences9 = df_freq_x_low10_2.groupby('Conceptno')['itemno'].apply(list).reset_index()

df_freq_y_top10_2 = df.groupby('Concept').apply(
    lambda x: x.nlargest(10, 'frequency_y')
).reset_index(drop=True)
df_sequences10 = df_freq_y_top10_2.groupby('Conceptno')['itemno'].apply(list).reset_index()

df_freq_y_low10_2 = df.groupby('Concept').apply(
    lambda x: x.nsmallest(10, 'frequency_y')
).reset_index(drop=True)
df_sequences11 = df_freq_y_low10_2.groupby('Conceptno')['itemno'].apply(list).reset_index()

df_dist_top10_2 = df.groupby('Concept').apply(
    lambda x: x.nlargest(10, 'Distinct')
).reset_index(drop=True)
df_sequences12 = df_dist_top10_2.groupby('Conceptno')['itemno'].apply(list).reset_index()

df_dist_low10_2 = df.groupby('Concept').apply(
    lambda x: x.nsmallest(10, 'Distinct')
).reset_index(drop=True)
df_sequences13 = df_dist_low10_2.groupby('Conceptno')['itemno'].apply(list).reset_index()


df_random_2 = df.groupby('Concept').apply(
    lambda x: x.sample(n=min(len(x), 10))
).reset_index(drop=True)
df_sequences14 = df_random_2.groupby('Conceptno')['itemno'].apply(list).reset_index()



In [208]:
data_frames = [df_sequences8, df_sequences9, df_sequences10, df_sequences11, df_sequences12, df_sequences13, df_sequences14]
new_names = ['freq_x_top10', 'freq_x_low10', 'freq_y_top10', 'freq_y_low10', 'dist_top10', 'dist_low10', 'random10']

for df, name in zip(data_frames, new_names):
    col_to_change = [c for c in df.columns if c != 'Conceptno'][0]
    df.rename(columns={col_to_change: name}, inplace=True)

from functools import reduce
df_merged_2 = reduce(lambda left, right: pd.merge(left, right, on='Conceptno', how='outer'), data_frames)

for col in new_names:
    df_merged_2[col] = df_merged_2[col].apply(lambda x: ', '.join(map(str, x)) if isinstance(x, list) else x)
with pd.ExcelWriter('pilot_features.xlsx', engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df_merged_2.to_excel(writer, sheet_name='sequences_itemno', index=False)
print



<function print(*args, sep=' ', end='\n', file=None, flush=False)>

In [209]:
import pandas as pd
import ast
import os

file_path = 'pilot_features.xlsx'
source_sheet = 'sequences_feature'
new_sheet_name = 'features_list'

target_columns = ['freq_x_top10', 'freq_x_low10', 'freq_y_top10', 'freq_y_low10', 'dist_top10', 'dist_low10', 'random10']

if os.path.exists(file_path):
    try:
        df = pd.read_excel(file_path, sheet_name=source_sheet)
        
        df.columns = df.columns.str.strip()
        print(f"Columns found in file: {df.columns.tolist()}")

        unique_features = set()
        
        for col in target_columns:
            if col in df.columns:
                print(f"--> Scanning column: '{col}'")
                
                sample = df[col].dropna().iloc[0] if not df[col].dropna().empty else "EMPTY"
                print(f"    Sample data in cell: {sample}")

                for cell_value in df[col]:
                    if pd.isna(cell_value): continue 
                    cell_str = str(cell_value).strip()
                    
                    try:
                        items = ast.literal_eval(cell_str)
                        if isinstance(items, list):
                            unique_features.update(items)
                        else:
                            unique_features.add(items)
                            
                    except (ValueError, SyntaxError):
                        clean_str = cell_str.replace('[', '').replace(']', '').replace("'", "")
                        items = [x.strip() for x in clean_str.split(',')]
                        unique_features.update(items)

            else:
                print(f"!! WARNING: Column '{col}' NOT found in Excel.")

        if len(unique_features) > 0:
            df_unique = pd.DataFrame(sorted(list(unique_features)), columns=['Feature'])
            
            with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
                df_unique.to_excel(writer, sheet_name=new_sheet_name, index=False)
                
            print(f"\nSUCCESS: Extracted {len(df_unique)} unique features.")
            print(f"Saved to sheet '{new_sheet_name}'.")
        else:
            print("\nERROR: No features were extracted. Please check the 'Sample data' printed above.")

    except Exception as e:
        print(f"An error occurred: {e}")
else:
    print(f"File not found: {file_path}")


Columns found in file: ['Conceptno', 'freq_x_top10', 'freq_x_low10', 'freq_y_top10', 'freq_y_low10', 'dist_top10', 'dist_low10', 'random10']
--> Scanning column: 'freq_x_top10'
    Sample data in cell: RÉSZE_LÁB, ILYEN_NAGY, ILYEN_BARNA, FAJTA_ÁLLAT, RÉSZE_UJJ, RÉSZE_FAROK, ILYEN_SÁRGA, ALFAJTA_ZÖLD, ILYEN_KICSI, RÉSZE_FEJ
--> Scanning column: 'freq_x_low10'
    Sample data in cell: HOL_ÁLLATKERT, ILYEN_ALACSONY, HOL_MESE, ILYEN_CUKI, RÉSZE_HANG, RÉSZE_NYAK, RÉSZE_TEST, ILYEN_CITROMSÁRGA, FAJTA_ÉLŐLÉNY, ILYEN_SZÍNES
--> Scanning column: 'freq_y_top10'
    Sample data in cell: RÉSZE_LÁB, FAJTA_ÁLLAT, ILYEN_BARNA, ILYEN_SÁRGA, RÉSZE_FEJ, RÉSZE_NYAK, ALFAJTA_KICSI, RÉSZE_SZEM, FAJTA_ÉLŐLÉNY, RÉSZE_TEST
--> Scanning column: 'freq_y_low10'
    Sample data in cell: RÉSZE_UJJ, RÉSZE_FAROK, ALFAJTA_ZÖLD, ILYEN_KÉK, HOL_OTTHON, TUD_MEGY, ILYEN_CITROMSÁRGA, ILYEN_CUKI, HOL_MESE, HOL_ÁLLATKERT
--> Scanning column: 'dist_top10'
    Sample data in cell: RÉSZE_FAROK, ILYEN_CUKI, HOL_ÁLLATKERT, ILYEN

In [210]:
import pandas as pd
import ast
import os

file_path = 'pilot_features.xlsx'
source_sheet = 'sequences_itemno'
new_sheet_name = 'itemno_list'

target_columns = ['freq_x_top10', 'freq_x_low10', 'freq_y_top10', 'freq_y_low10', 'dist_top10', 'dist_low10', 'random10']

if os.path.exists(file_path):
    try:
        df = pd.read_excel(file_path, sheet_name=source_sheet)
        
        df.columns = df.columns.str.strip()
        print(f"Columns found in file: {df.columns.tolist()}")

        unique_itemno = set()
        
        for col in target_columns:
            if col in df.columns:
                print(f"--> Scanning column: '{col}'")
                
                sample = df[col].dropna().iloc[0] if not df[col].dropna().empty else "EMPTY"
                print(f"    Sample data in cell: {sample}")

                for cell_value in df[col]:
                    if pd.isna(cell_value): continue 
                    cell_str = str(cell_value).strip()
                    
                    try:
                        items = ast.literal_eval(cell_str)
                        if isinstance(items, list):
                            unique_itemno.update(items)
                        else:
                            unique_itemno.add(items)
                            
                    except (ValueError, SyntaxError):
                        clean_str = cell_str.replace('[', '').replace(']', '').replace("'", "")
                        items = [x.strip() for x in clean_str.split(',')]
                        unique_itemno.update(items)

            else:
                print(f"!! WARNING: Column '{col}' NOT found in Excel.")

        if len(unique_itemno) > 0:
            df_unique = pd.DataFrame(sorted(list(unique_itemno)), columns=['itemno'])
            
            with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
                df_unique.to_excel(writer, sheet_name=new_sheet_name, index=False)
                
            print(f"\nSUCCESS: Extracted {len(df_unique)} unique itemno.")
            print(f"Saved to sheet '{new_sheet_name}'.")
        else:
            print("\nERROR: No itemno were extracted. Please check the 'Sample data' printed above.")

    except Exception as e:
        print(f"An error occurred: {e}")
else:
    print(f"File not found: {file_path}")


Columns found in file: ['Conceptno', 'freq_x_top10', 'freq_x_low10', 'freq_y_top10', 'freq_y_low10', 'dist_top10', 'dist_low10', 'random10']
--> Scanning column: 'freq_x_top10'
    Sample data in cell: f00189, f00968, f00840, f00660, f00485, f01755, f00863, f00843, f00966, f01056
--> Scanning column: 'freq_x_low10'
    Sample data in cell: f01505, f01759, f01076, f00976, f00974, f00176, f00444, f00867, f00635, f00851
--> Scanning column: 'freq_y_top10'
    Sample data in cell: f00189, f00660, f00840, f00863, f01056, f00176, f00969, f00483, f00635, f00444
--> Scanning column: 'freq_y_low10'
    Sample data in cell: f00485, f01755, f00843, f00861, f00774, f01537, f00867, f00976, f01076, f01505
--> Scanning column: 'dist_top10'
    Sample data in cell: f01755, f00976, f01505, f01759, f00483, f01537, f00867, f00974, f00176, f00660
--> Scanning column: 'dist_low10'
    Sample data in cell: f00968, f00840, f00969, f00774, f00966, f00843, f00864, f00189, f00863, f01056
--> Scanning column: 'r

In [211]:
import pandas as pd
import os

file_path = 'pilot_features.xlsx'
sheet1_name = 'features_list'  
sheet2_name = 'sorted_by_features'    
col_sheet1_key = 'Feature'
col_sheet2_key = 'Feature'
col_sheet2_val = 'itemno'

if os.path.exists(file_path):
    try:
        df1 = pd.read_excel(file_path, sheet_name=sheet1_name)
        df2 = pd.read_excel(file_path, sheet_name=sheet2_name)

        df1[col_sheet1_key] = df1[col_sheet1_key].astype(str).str.strip()
        df2[col_sheet2_key] = df2[col_sheet2_key].astype(str).str.strip()

        source_df = df2[[col_sheet2_key, col_sheet2_val]].drop_duplicates(subset=[col_sheet2_key])

        merged_df = pd.merge(
            df1, 
            source_df, 
            left_on=col_sheet1_key, 
            right_on=col_sheet2_key, 
            how='left'
        )

        if col_sheet2_key in merged_df.columns and col_sheet2_key != col_sheet1_key:
            merged_df = merged_df.drop(columns=[col_sheet2_key])

        with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
            merged_df.to_excel(writer, sheet_name=sheet1_name, index=False)

        print(f"Success! Added '{col_sheet2_val}' to '{sheet1_name}'.")
        print(f"Rows in Sheet 1: {len(merged_df)}")
        
        missing_count = merged_df[col_sheet2_val].isna().sum()
        if missing_count > 0:
            print(f"Warning: {missing_count} features in Sheet 1 did not find a match in Sheet 2.")

    except PermissionError:
        print(f"Error: Please close '{file_path}' in Excel and try again.")
    except Exception as e:
        print(f"An error occurred: {e}")
else:
    print(f"File '{file_path}' not found.")


Success! Added 'itemno' to 'features_list'.
Rows in Sheet 1: 181


In [212]:
import pandas as pd
import os

file_path = 'pilot_features.xlsx'
sheet1_name = 'itemno_list'  
sheet2_name = 'sorted_by_features'   

col_sheet1_key = 'itemno'
col_sheet2_key = 'itemno'
col_sheet2_val = 'Feature'

if os.path.exists(file_path):
    try:
        df1 = pd.read_excel(file_path, sheet_name=sheet1_name)
        df2 = pd.read_excel(file_path, sheet_name=sheet2_name)

        df1[col_sheet1_key] = df1[col_sheet1_key].astype(str).str.strip()
        df2[col_sheet2_key] = df2[col_sheet2_key].astype(str).str.strip()
        source_df = df2[[col_sheet2_key, col_sheet2_val]].drop_duplicates(subset=[col_sheet2_key])
        merged_df = pd.merge(
            df1, 
            source_df, 
            left_on=col_sheet1_key, 
            right_on=col_sheet2_key, 
            how='left'
        )

        if col_sheet2_key in merged_df.columns and col_sheet2_key != col_sheet1_key:
            merged_df = merged_df.drop(columns=[col_sheet2_key])
        with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
            merged_df.to_excel(writer, sheet_name=sheet1_name, index=False)

        print(f"Success! Added '{col_sheet2_val}' to '{sheet1_name}'.")
        print(f"Rows in Sheet 1: {len(merged_df)}")
        
        missing_count = merged_df[col_sheet2_val].isna().sum()
        if missing_count > 0:
            print(f"Warning: {missing_count} features in Sheet 1 did not find a match in Sheet 2.")

    except PermissionError:
        print(f"Error: Please close '{file_path}' in Excel and try again.")
    except Exception as e:
        print(f"An error occurred: {e}")
else:
    print(f"File '{file_path}' not found.")


Success! Added 'Feature' to 'itemno_list'.
Rows in Sheet 1: 182


In [213]:
import pandas as pd
file_path='pilot_features.xlsx'
df = pd.read_excel(file_path, sheet_name='features_list')
df = df[['itemno', 'Feature']]
df = df.sort_values(by='itemno', ascending=True)

with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df.to_excel(writer, sheet_name='features_list', index=False)
df.head()


,itemno,Feature
71,f00022,ILYEN_PICI
85,f00047,KI_GYEREK
73,f00071,ILYEN_PUHA
20,f00095,FAJTA_RUHADARAB
174,f00110,TUD_SEGÍT


In [214]:
import pandas as pd
file_path = 'pilot_features.xlsx'
sheet1_name = 'features_list'  
sheet2_name = 'itemno_list'  

try:
    df_a = pd.read_excel(file_path, sheet_name=sheet1_name, dtype=str)
    df_b = pd.read_excel(file_path, sheet_name=sheet2_name, dtype=str)

    df_a = df_a.fillna('')
    df_b = df_b.fillna('')

    if df_a.equals(df_b):
        print(f"✅ SUCCESS: '{sheet1_name}' and '{sheet2_name}' are EXACTLY the same.")
    else:
        print(f"❌ MISMATCH: The sheets are different.")
        
        if df_a.shape != df_b.shape:
            print(f"   - Shape mismatch: {sheet1_name} is {df_a.shape}, {sheet2_name} is {df_b.shape}")
        
        elif list(df_a.columns) != list(df_b.columns):
            print(f"   - Column name mismatch.")
            print(f"     {sheet1_name}: {list(df_a.columns)}")
            print(f"     {sheet2_name}: {list(df_b.columns)}")
            
        else:
            print("   - Dimensions and Columns match, but cell values differ.")

            diff = df_a.compare(df_b)
            
            print(f"   - Found {len(diff)} rows with differences.")
            print("   - Sample of differences (self=Sheet A, other=Sheet B):")
            print(diff.head())

except Exception as e:
    print(f"An error occurred: {e}")


❌ MISMATCH: The sheets are different.
   - Shape mismatch: features_list is (181, 2), itemno_list is (182, 2)


In [ ]:
import pandas as pd

file_path = 'pilot_features.xlsx'
sheet1_name = 'features_list'
sheet2_name = 'itemno_list'

try:
    df1 = pd.read_excel(file_path, sheet_name=sheet1_name, dtype=str).fillna('')
    df2 = pd.read_excel(file_path, sheet_name=sheet2_name, dtype=str).fillna('')
    if list(df1.columns) != list(df2.columns):
        print("Warning: Column names were different. Renaming Sheet 2 columns to match Sheet 1 for comparison.")
        df2.columns = df1.columns

    merged_df = df1.merge(df2, how='outer', indicator=True)

    missing_in_sheet2 = merged_df[merged_df['_merge'] == 'left_only'].drop(columns=['_merge'])
    missing_in_sheet1 = merged_df[merged_df['_merge'] == 'right_only'].drop(columns=['_merge'])

    print(f"--- COMPARISON REPORT ---")
    
    if missing_in_sheet2.empty and missing_in_sheet1.empty:
        print("✅ Perfect Match! No rows are missing.")
    else:
        if not missing_in_sheet2.empty:
            print(f"\n❌ Found {len(missing_in_sheet2)} rows in '{sheet1_name}' that are MISSING from '{sheet2_name}':")
            print(missing_in_sheet2.head())
        
        if not missing_in_sheet1.empty:
            print(f"\n❌ Found {len(missing_in_sheet1)} rows in '{sheet2_name}' that are MISSING from '{sheet1_name}':")
            print(missing_in_sheet1.head())

except Exception as e:
    print(f"An error occurred: {e}")


--- COMPARISON REPORT ---

❌ Found 1 rows in 'itemno_list' that are MISSING from 'features_list':
    itemno        Feature
80  f00847  ILYEN_SZÍNES 


In [ ]:
import pandas as pd
import os

file_path = 'pilot_features.xlsx'
sheet_name = 'sequences_itemno'

columns_to_process = ['freq_x_top10', 'freq_x_low10', 'freq_y_top10', 'freq_y_low10', 'dist_top10', 'dist_low10', 'random10']  # Example: replace with your actual column names

id_column = 'Conceptno' 

if not os.path.exists(file_path):
    raise FileNotFoundError(f"The file '{file_path}' was not found.")

df = pd.read_excel(file_path, sheet_name=sheet_name)

df.columns = df.columns.str.strip()

processed_dfs = []

for col_name in columns_to_process:
    if col_name not in df.columns:
        print(f"Warning: Column '{col_name}' not found in Excel. Skipping.")
        continue
    
    print(f"Processing column: {col_name}...")
    
    split_data = df[col_name].str.split(',', expand=True)
    
    split_data = split_data.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
    
    num_cols = split_data.shape[1]
    split_data.columns = [f'stimulus {i+1}' for i in range(num_cols)]
    
    split_data['sequence_type'] = col_name
    
    split_data['original_string'] = df[col_name]
    
    if id_column and id_column in df.columns:
        split_data[id_column] = df[id_column]
    else:
        split_data['row_index'] = df.index

    processed_dfs.append(split_data)

if processed_dfs:
    final_df = pd.concat(processed_dfs, ignore_index=True)
    cols = list(final_df.columns)
    priority_cols = ['sequence_type', 'original_string']
    if id_column and id_column in df.columns:
        priority_cols.insert(0, id_column)
        
    for col in priority_cols:
        if col in cols:
            cols.insert(0, cols.pop(cols.index(col)))
            
    final_df = final_df[cols]

    try:
        with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
            final_df.to_excel(writer, sheet_name='stimuli', index=False)
        print("Success! Data saved to sheet 'stimuli'.")
        print(final_df.head())
    except PermissionError:
        print(f"Error: Could not write to '{file_path}'. Please close the Excel file and try again.")
else:
    print("No columns were processed. Check your column names.")


Processing column: freq_x_top10...
Processing column: freq_x_low10...
Processing column: freq_y_top10...
Processing column: freq_y_low10...
Processing column: dist_top10...
Processing column: dist_low10...
Processing column: random10...
Success! Data saved to sheet 'stimuli'.
                                     original_string sequence_type Conceptno  \
0  f00189, f00968, f00840, f00660, f00485, f01755...  freq_x_top10       a05   
1  f00189, f00968, f00840, f00772, f00660, f01755...  freq_x_top10       a16   
2  f00189, f00968, f00840, f00660, f01755, f00188...  freq_x_top10       a19   
3  f00840, f00862, f00532, f00843, f01588, f00440...  freq_x_top10       b02   
4  f00417, f00968, f00485, f01327, f02502, f00457...  freq_x_top10       b10   

  stimulus 1 stimulus 2 stimulus 3 stimulus 4 stimulus 5 stimulus 6  \
0     f00189     f00968     f00840     f00660     f00485     f01755   
1     f00189     f00968     f00840     f00772     f00660     f01755   
2     f00189     f00968     f

In [217]:
import pandas as pd

file_path = 'pilot_features.xlsx' 
sheet_name = 'stimuli'
df = pd.read_excel(file_path, sheet_name=sheet_name)

if 'sequence_ID' in df.columns:
    df = df.drop(columns=['sequence_ID'])

df.insert(loc=0, column='sequence_ID', value=range(1, len(df) + 1))

print(df.head())


with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df.to_excel(writer, sheet_name=sheet_name, index=False)


   sequence_ID                                    original_string  \
0            1  f00189, f00968, f00840, f00660, f00485, f01755...   
1            2  f00189, f00968, f00840, f00772, f00660, f01755...   
2            3  f00189, f00968, f00840, f00660, f01755, f00188...   
3            4  f00840, f00862, f00532, f00843, f01588, f00440...   
4            5  f00417, f00968, f00485, f01327, f02502, f00457...   

  sequence_type Conceptno stimulus 1 stimulus 2 stimulus 3 stimulus 4  \
0  freq_x_top10       a05     f00189     f00968     f00840     f00660   
1  freq_x_top10       a16     f00189     f00968     f00840     f00772   
2  freq_x_top10       a19     f00189     f00968     f00840     f00660   
3  freq_x_top10       b02     f00840     f00862     f00532     f00843   
4  freq_x_top10       b10     f00417     f00968     f00485     f01327   

  stimulus 5 stimulus 6 stimulus 7 stimulus 8 stimulus 9 stimulus 10  
0     f00485     f01755     f00863     f00843     f00966      f01056  
1   

In [ ]:
import pandas as pd
import os

file_path = 'pilot_features.xlsx'
sheet_name = 'stimuli'

df = pd.read_excel(file_path, sheet_name=sheet_name)

stimulus_cols = [f'stimulus {i+1}' for i in range(10)]

def format_stimulus_path(val):
    if pd.isna(val):
        return val
    
    clean_val = str(val).strip(" '\"")
    
    if not clean_val:
        return None
    return f"stimuli/{clean_val}.png"

for col in stimulus_cols:
    if col in df.columns:
        df[col] = df[col].apply(format_stimulus_path)
    else:
        print(f"Warning: Column '{col}' not found in the Excel file.")

with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df.to_excel(writer, sheet_name='stimuli', index=False)

print(f"Processing complete")
print(df[stimulus_cols].head())


Processing complete
           stimulus 1          stimulus 2          stimulus 3  \
0  stimuli/f00189.png  stimuli/f00968.png  stimuli/f00840.png   
1  stimuli/f00189.png  stimuli/f00968.png  stimuli/f00840.png   
2  stimuli/f00189.png  stimuli/f00968.png  stimuli/f00840.png   
3  stimuli/f00840.png  stimuli/f00862.png  stimuli/f00532.png   
4  stimuli/f00417.png  stimuli/f00968.png  stimuli/f00485.png   

           stimulus 4          stimulus 5          stimulus 6  \
0  stimuli/f00660.png  stimuli/f00485.png  stimuli/f01755.png   
1  stimuli/f00772.png  stimuli/f00660.png  stimuli/f01755.png   
2  stimuli/f00660.png  stimuli/f01755.png  stimuli/f00188.png   
3  stimuli/f00843.png  stimuli/f01588.png  stimuli/f00440.png   
4  stimuli/f01327.png  stimuli/f02502.png  stimuli/f00457.png   

           stimulus 7          stimulus 8          stimulus 9  \
0  stimuli/f00863.png  stimuli/f00843.png  stimuli/f00966.png   
1  stimuli/f00188.png  stimuli/f00192.png  stimuli/f01327.png   
2  

In [ ]:
import pandas as pd
import os

file_path = 'pilot_features.xlsx'
sheet_name = 'stimuli' 
output_folder = 'stimuli'  

columns_to_exclude = ['original_string']

if not os.path.exists(file_path):
    raise FileNotFoundError(f"File not found: {file_path}")

df = pd.read_excel(file_path, sheet_name=sheet_name)

os.makedirs(output_folder, exist_ok=True)
print(f"Saving files to folder: {output_folder}/")

count = 0
for index, row in df.iterrows():
    if 'sequence_ID' in row and pd.notnull(row['sequence_ID']):
        seq_id = str(int(row['sequence_ID'])) if isinstance(row['sequence_ID'], (int, float)) else str(row['sequence_ID'])
    else:
        seq_id = f"row_{index}"

    filename = f"sequence{seq_id}.csv"
    full_path = os.path.join(output_folder, filename)

    single_row_df = df.iloc[[index]].copy()
    single_row_df = single_row_df.drop(columns=columns_to_exclude, errors='ignore')

    single_row_df.to_csv(full_path, index=False)
    count += 1

print(f"Success! Generated {count} CSV files.")


Saving files to folder: stimuli/
Success! Generated 210 CSV files.
